# 09 · Recommenders, and the Loop They Sit In

The lesson's claim is that a recommender's central difficulty is not the algorithm but
the fact that it trains on data it produced. This builds a small collaborative filter,
watches it work, and then watches the loop close around it.

Stdlib only, seeded, no model calls.

In [1]:
import math, random
from collections import Counter, defaultdict
random.seed(9)

USERS, ITEMS = 300, 400
# Three latent tastes. Users belong to one; items serve one. Nobody tells the model.
TASTES = 3
user_taste = [random.randrange(TASTES) for _ in range(USERS)]
item_taste = [random.randrange(TASTES) for _ in range(ITEMS)]
# Item quality, so popularity is not purely an artefact of taste.
item_quality = [random.uniform(0.3, 1.0) for _ in range(ITEMS)]

def would_like(u, i):
    """Ground truth we can check against, and that no recommender ever sees."""
    base = 0.75 if user_taste[u] == item_taste[i] else 0.10
    return base * item_quality[i]

print(f"{USERS} users, {ITEMS} items, {TASTES} hidden tastes")
print(f"if every user rated every item the matrix would hold {USERS*ITEMS:,} cells")

300 users, 400 items, 3 hidden tastes
if every user rated every item the matrix would hold 120,000 cells


## The matrix people actually have

Users see a fraction of the catalogue and interact with a fraction of that. Everything
below works from this — and note what it does *not* contain: any record of what was
shown and passed over.

In [2]:
def simulate_exposure(slate_for, rounds=4, slate=5):
    """Each round every user is shown a slate and clicks probabilistically."""
    clicks = defaultdict(set)
    shown = defaultdict(set)
    for _ in range(rounds):
        for u in range(USERS):
            for i in slate_for(u, clicks[u]):
                shown[u].add(i)
                if random.random() < would_like(u, i):
                    clicks[u].add(i)
    return clicks, shown

def random_slate(u, seen, k=5):
    return random.sample(range(ITEMS), k)

clicks, shown = simulate_exposure(random_slate)
n_clicks = sum(len(v) for v in clicks.values())
print(f"interactions recorded : {n_clicks:,}")
print(f"matrix density        : {100*n_clicks/(USERS*ITEMS):.1f}%")
print(f"users with fewer than 3 interactions: "
      f"{sum(1 for u in range(USERS) if len(clicks[u]) < 3)}")

interactions recorded : 1,271
matrix density        : 1.1%
users with fewer than 3 interactions: 50


## A collaborative filter, in twenty lines

No factorisation, no library. For a user, find similar users by overlap, and score
items by what those neighbours engaged with. It is the simplest thing that deserves
the name and it is enough to show every failure the lesson describes.

In [3]:
def jaccard(a, b):
    return len(a & b) / len(a | b) if (a | b) else 0.0

def recommend(u, history, k=5, neighbours=20):
    sims = sorted(((jaccard(history, clicks[v]), v) for v in range(USERS) if v != u),
                  reverse=True)[:neighbours]
    score = Counter()
    for s, v in sims:
        if s <= 0:
            continue
        for i in clicks[v] - history:
            score[i] += s
    return [i for i, _ in score.most_common(k)]

def precision_at_k(recommender, k=5, trials=150):
    hits = total = 0
    for u in random.sample(range(USERS), trials):
        for i in recommender(u, clicks[u], k=k):
            hits += 1 if user_taste[u] == item_taste[i] else 0
            total += 1
    return hits / total if total else 0.0

print(f"precision@5, collaborative filter : {precision_at_k(recommend):.2f}")
print(f"precision@5, random baseline      : {precision_at_k(lambda u,h,k=5: random.sample(range(ITEMS),k)):.2f}")
print("\nit works — it recovered the hidden tastes from interactions alone,")
print("having been told nothing about users or items")

precision@5, collaborative filter : 0.66
precision@5, random baseline      : 0.35

it works — it recovered the hidden tastes from interactions alone,
having been told nothing about users or items


## Cold start, which is not a corner case

In [4]:
cold = [u for u in range(USERS) if len(clicks[u]) <= 1]
warm = [u for u in range(USERS) if len(clicks[u]) >= 8]

def precision_for(users, k=5):
    hits = total = 0
    for u in users:
        for i in recommend(u, clicks[u], k=k):
            hits += 1 if user_taste[u] == item_taste[i] else 0
            total += 1
    return (hits / total if total else 0.0), len(users)

pc, nc = precision_for(cold[:60])
pw, nw = precision_for(warm[:60])
print(f"users with <=1 interaction ({nc:>3}): precision@5 = {pc:.2f}")
print(f"users with >=8 interactions ({nw:>3}): precision@5 = {pw:.2f}")

never_clicked = [i for i in range(ITEMS) if not any(i in c for c in clicks.values())]
print(f"\nitems nobody has ever clicked: {len(never_clicked)} of {ITEMS}")
print("collaborative filtering cannot recommend any of them, because it only knows items")
print("through the users who touched them — and nobody has")

users with <=1 interaction ( 12): precision@5 = 0.40
users with >=8 interactions ( 11): precision@5 = 0.69

items nobody has ever clicked: 27 of 400
collaborative filtering cannot recommend any of them, because it only knows items
through the users who touched them — and nobody has


## Now close the loop

Everything so far used randomly chosen slates, which is not how a deployed system
behaves. Let the recommender choose what gets shown, feed the resulting clicks back in,
and repeat — which is the arrangement every production recommender is actually in.

In [5]:
def run_loop(rounds=8, slate=5, explore=0.0):
    """Recommend, observe, retrain on what was observed. Repeat."""
    global clicks
    clicks = defaultdict(set)
    for u in range(USERS):                      # a small warm-up history
        for i in random.sample(range(ITEMS), 4):
            if random.random() < would_like(u, i):
                clicks[u].add(i)
    trace = []
    for r in range(rounds):
        exposed = Counter()
        for u in range(USERS):
            rec = recommend(u, clicks[u], k=slate)
            if len(rec) < slate:
                rec += random.sample(range(ITEMS), slate - len(rec))
            if explore:
                keep = int(slate * (1 - explore))
                rec = rec[:keep] + random.sample(range(ITEMS), slate - keep)
            for i in rec:
                exposed[i] += 1
                if random.random() < would_like(u, i):
                    clicks[u].add(i)
        coverage = len(exposed) / ITEMS
        top10 = sum(c for _, c in exposed.most_common(max(1, ITEMS // 10))) / sum(exposed.values())
        trace.append((r + 1, coverage, top10, sum(len(v) for v in clicks.values())))
    return trace

print(f"{'round':>6}{'catalogue coverage':>20}{'share in top 10% of items':>28}{'clicks':>9}")
for r, cov, top, cl in run_loop():
    print(f"{r:>6}{100*cov:>19.0f}%{100*top:>27.0f}%{cl:>9}")

 round  catalogue coverage   share in top 10% of items   clicks


     1                 98%                         22%      560
     2                 89%                         29%     1001
     3                 78%                         39%     1504
     4                 64%                         43%     2020
     5                 55%                         41%     2501
     6                 55%                         39%     2956
     7                 56%                         39%     3353
     8                 56%                         42%     3708


Read the columns together. Clicks rise steadily — engagement is improving, and a
dashboard showing only that column would report a system getting better every round.

Coverage falls, and the share of exposure going to a handful of items climbs. The
catalogue is narrowing in front of every user, and the clicks are rising *because* it
is narrowing: the system has found the items that reliably get clicked and is showing
them to more people. Nothing in the objective asked for this.

## The fix the lesson names, measured

In [6]:
print(f"{'exploration':>12}{'final coverage':>17}{'final concentration':>21}{'clicks':>9}")
for e in (0.0, 0.1, 0.2, 0.4):
    r, cov, top, cl = run_loop(explore=e)[-1]
    print(f"{100*e:>11.0f}%{100*cov:>16.0f}%{100*top:>20.0f}%{cl:>9}")

print("\nexploration bought coverage. read the clicks column before assuming it cost any —")
print("on this run it did not, which means the greedy policy was leaving engagement on")
print("the table as well as narrowing the catalogue. one seeded run is not proof that")
print("exploration is free; it is proof that assuming it is expensive needs checking.")

 exploration   final coverage  final concentration   clicks


          0%              65%                  39%     3751


         10%              81%                  31%     3909


         20%              79%                  33%     4170


         40%              85%                  38%     3880

exploration bought coverage. read the clicks column before assuming it cost any —
on this run it did not, which means the greedy policy was leaving engagement on
the table as well as narrowing the catalogue. one seeded run is not proof that
exploration is free; it is proof that assuming it is expensive needs checking.


## What to carry away

- **A collaborative filter recovers structure nobody described**, from interactions
  alone. That is genuinely impressive and it is not the hard part.
- **Cold start is most of your users**, not an edge case — measure precision on the
  sparse ones separately or the average will hide them.
- **The loop closes on its own.** Coverage falls and concentration rises while
  engagement improves, and only the first two columns can tell you.
- **Exploration buys the one thing self-generated data cannot contain**: evidence
  about what you never showed. Whether it *costs* engagement is an empirical question
  and the answer here was no — the greedy policy had concentrated onto items that were
  reliably clicked rather than best-matched, so widening the slate found better ones.
  Do not carry that result to your own system; carry the habit of checking it.

The guardrail argument from the measurement lesson applies here with unusual force,
because the primary metric improves throughout the failure.